# 01 — Build Embeddings + FAISS Index (Drive-cached)
**Goal:** Create semantic retrieval assets for grouped batching.

**This notebook does:**
1. Loads `bootstrap_manifest.json`
2. Loads a dataset from Drive (default: `dolly_small_1k`)
3. Generates sentence embeddings for the **instruction** field
4. Builds a FAISS index (cosine via inner product on normalized vectors)
5. Saves embeddings + index + id mapping back to Drive

**Idempotent:** if files exist, it will skip recomputation unless you set `FORCE_REBUILD=True`.


In [1]:
# --- 0) Mount Drive ---
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")


Mounted at /content/drive


In [3]:
!pip install -q faiss-cpu sentence-transformers datasets tqdm  # faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.9 MB/s eta 0:00:00


In [ ]:
# --- 1) Imports + Small Utilities ---
import os
import json
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
from tqdm.auto import tqdm

import faiss
from datasets import load_from_disk
from sentence_transformers import SentenceTransformer

def _exists_nonempty(path: str) -> bool:
    return os.path.exists(path) and (os.path.isdir(path) and len(os.listdir(path)) > 0 or os.path.isfile(path) and os.path.getsize(path) > 0)

def _now_utc() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def _safe_makedirs(path: str):
    os.makedirs(path, exist_ok=True)


In [ ]:
# --- 2) Load Bootstrap Manifest (single source of truth for paths) ---
ROOT = '/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research'
MANIFEST_PATH = f'{ROOT}/manifests/bootstrap_manifest.json'

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f'bootstrap_manifest.json not found at: {MANIFEST_PATH}\n'
        'Run notebooks/00_bootstrap_downloads.ipynb first.'
    )

with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

DIRS = manifest['dirs']
print('Loaded manifest created_at_utc:', manifest.get('created_at_utc'))
print('ROOT:', manifest.get('root', ROOT))
print('shared_datasets_raw:', DIRS['shared_datasets_raw'])
print('shared_indexes_faiss:', DIRS['shared_indexes_faiss'])


## Config
Set which dataset + embedding model to use.

- **Dataset**: `dolly_small_1k` is ideal for fast validation.
- **Embedding model**: `all-MiniLM-L6-v2` is fast and strong for semantic similarity.


In [ ]:
# --- 3) Config ---
FORCE_REBUILD = False  # set True to recompute embeddings/index even if cached

# Choose dataset saved by bootstrap notebook
DATASET_NAME = 'dolly_small_1k'   # alternatives: 'dolly_15k'
SPLIT_NAME = 'train'

# Choose embedding model directory saved by bootstrap
EMBED_MODEL_DIRNAME = 'all-MiniLM-L6-v2'

# Which field to embed (Dolly has 'instruction')
TEXT_FIELD = 'instruction'

# Embedding settings
BATCH_SIZE = 64
MAX_TEXT_CHARS = 2000  # guardrail to avoid very long strings

# FAISS settings
USE_GPU_FAISS = False  # keep False on Colab CPU runtime; True only if you install faiss-gpu


In [ ]:
# --- 4) Resolve Paths ---
dataset_path = f"{DIRS['shared_datasets_raw']}/{DATASET_NAME}"
embed_model_path = f"{DIRS['shared_models_embed']}/{EMBED_MODEL_DIRNAME}"

# Output bundle name ties dataset+model together
bundle_name = f"{DATASET_NAME}__{EMBED_MODEL_DIRNAME}"

bundle_dir = f"{DIRS['shared_indexes_faiss']}/{bundle_name}"
_safe_makedirs(bundle_dir)

emb_path = f"{bundle_dir}/embeddings.npy"          # float32 [N, D]
meta_path = f"{bundle_dir}/meta.json"             # fields, dims, etc.
id_map_path = f"{bundle_dir}/id_map.json"         # row index -> dataset id or row index
faiss_index_path = f"{bundle_dir}/faiss.index"    # faiss index

print('Dataset:', dataset_path)
print('Embed model:', embed_model_path)
print('Bundle dir:', bundle_dir)


Dataset: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/datasets/raw/dolly_small_1k
Embed model: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/models/embeddings/all-MiniLM-L6-v2
Bundle dir: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2


## Load dataset
We load the dataset from Drive and extract the text field for embeddings.


In [ ]:
# --- 5) Load Dataset ---
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f'Dataset not found at: {dataset_path}')

ds = load_from_disk(dataset_path)
if SPLIT_NAME not in ds:
    raise KeyError(f"Split '{SPLIT_NAME}' not found. Available: {list(ds.keys())}")

split = ds[SPLIT_NAME]
if TEXT_FIELD not in split.column_names:
    raise KeyError(f"Field '{TEXT_FIELD}' not found. Available: {split.column_names}")

print('Rows:', len(split))
print('Columns:', split.column_names)
print('Example instruction:', split[0][TEXT_FIELD])


Rows: 1000
Columns: ['instruction', 'context', 'response', 'category']
Example instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?


## Compute embeddings (cached)
- Uses SentenceTransformers
- Normalizes vectors for cosine similarity
- Saves to Drive as `embeddings.npy`


In [ ]:
# --- 6) Load Embedding Model ---
if not os.path.exists(embed_model_path):
    raise FileNotFoundError(f'Embedding model not found at: {embed_model_path}')

embedder = SentenceTransformer(embed_model_path)
embedder.max_seq_length = 256  # conservative, keeps compute stable
print('Loaded embedder:', EMBED_MODEL_DIRNAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedder: all-MiniLM-L6-v2


In [ ]:
# --- 7) Build Embeddings (idempotent) ---
def _clean_text(x: str) -> str:
    if x is None:
        return ''
    s = str(x)
    s = s.replace('\u0000', '')
    if len(s) > MAX_TEXT_CHARS:
        s = s[:MAX_TEXT_CHARS]
    return s

need_embeddings = FORCE_REBUILD or (not _exists_nonempty(emb_path)) or (not _exists_nonempty(meta_path))

if need_embeddings:
    texts = [_clean_text(t) for t in split[TEXT_FIELD]]

    # Encode in batches; output np.float32
    all_vecs = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding'):
        batch = texts[i:i+BATCH_SIZE]
        vec = embedder.encode(
            batch,
            batch_size=len(batch),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,  # cosine-ready
        ).astype('float32')
        all_vecs.append(vec)

    X = np.vstack(all_vecs).astype('float32')

    # Persist
    np.save(emb_path, X)
    meta = {
        'created_at_utc': _now_utc(),
        'dataset_name': DATASET_NAME,
        'split': SPLIT_NAME,
        'text_field': TEXT_FIELD,
        'embed_model': EMBED_MODEL_DIRNAME,
        'n_rows': int(X.shape[0]),
        'dim': int(X.shape[1]),
        'normalized': True,
        'dtype': str(X.dtype),
    }
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)

    # Id map: try to keep stable identifiers if dataset has an 'id' column, else row index
    if 'id' in split.column_names:
        id_map = [str(x) for x in split['id']]
    else:
        id_map = [str(i) for i in range(len(split))]
    with open(id_map_path, 'w', encoding='utf-8') as f:
        json.dump({'created_at_utc': _now_utc(), 'id_map': id_map}, f)

    print('Saved embeddings:', emb_path)
    print('Saved meta:', meta_path)
    print('Saved id map:', id_map_path)
else:
    print('Embeddings already cached. Skipping (set FORCE_REBUILD=True to recompute).')


Embedding:   0%|          | 0/16 [00:00<?, ?it/s]

Saved embeddings: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/embeddings.npy
Saved meta: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/meta.json
Saved id map: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/id_map.json


## Build FAISS index (cached)
We use **inner product** on **normalized embeddings**, equivalent to cosine similarity.


In [ ]:
# --- 8) Build FAISS Index (idempotent) ---
need_index = FORCE_REBUILD or (not _exists_nonempty(faiss_index_path))

# Load embeddings from disk
X = np.load(emb_path)
if X.dtype != np.float32:
    X = X.astype('float32')

if need_index:
    d = X.shape[1]
    index = faiss.IndexFlatIP(d)  # cosine via normalized vectors
    index.add(X)

    faiss.write_index(index, faiss_index_path)
    print('Saved FAISS index:', faiss_index_path)
    print('Index ntotal:', index.ntotal)
else:
    index = faiss.read_index(faiss_index_path)
    print('Loaded cached FAISS index:', faiss_index_path)
    print('Index ntotal:', index.ntotal)


Saved FAISS index: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/faiss.index
Index ntotal: 1000


## Sanity check: nearest neighbors
Pick a few random instructions and inspect the nearest retrieved neighbors.


In [ ]:
# --- 9) Nearest Neighbor Inspection ---
import random

with open(id_map_path, 'r', encoding='utf-8') as f:
    id_map = json.load(f)['id_map']

def search_neighbors(row_idx: int, k: int = 6) -> List[Tuple[int, float]]:
    q = X[row_idx:row_idx+1]
    scores, idxs = index.search(q, k)
    return list(zip(idxs[0].tolist(), scores[0].tolist()))

random.seed(42)
for row_idx in random.sample(range(len(split)), k=min(3, len(split))):
    print('\n' + '='*90)
    print('QUERY idx:', row_idx, 'id:', id_map[row_idx])
    print('TEXT:', split[row_idx][TEXT_FIELD])

    nn = search_neighbors(row_idx, k=6)
    print('\nTop neighbors:')
    for j, score in nn:
        print(f'  - idx={j:5d}  score={score: .4f}  id={id_map[j]}')
        # show short preview
        t = split[j][TEXT_FIELD]
        t = (t[:160] + '…') if len(t) > 160 else t
        print('    ', t)



QUERY idx: 654 id: 654
TEXT: Identify which car manufacturer is Italian or American: Lancia, Lincoln

Top neighbors:
  - idx=  654  score= 1.0000  id=654
     Identify which car manufacturer is Italian or American: Lancia, Lincoln
  - idx=  272  score= 0.8817  id=272
     Identify which car manufacturer is Italian or American: Lancia, Tesla
  - idx=  176  score= 0.5641  id=176
     Identify which car manufacturer is Chinese or American: BAIC, GMC
  - idx=  960  score= 0.5628  id=960
     Identify which car manufacturer is British or American: Aston Martin, Cadillac
  - idx=  443  score= 0.5466  id=443
     Identify which car manufacturer is Japanese or American: Dodge, Subaru
  - idx=  908  score= 0.4124  id=908
     Tell me whether these companies are banks or car manufacturers: Chase, Wells Fargo, GMC, Ford, Tesla, Truist

QUERY idx: 114 id: 114
TEXT: Tell me whether these are a solid, liquid, or gas

Top neighbors:
  - idx=  114  score= 1.0000  id=114
     Tell me whether these are

## Output summary
At this point you have a Drive-cached retrieval bundle:
- `embeddings.npy`
- `faiss.index`
- `id_map.json`
- `meta.json`

Next notebook will use these to construct **grouped mini-batches**.
